In [42]:
# setup from 01_tdbrain_eda.ipynb

from pathlib import Path
from collections import defaultdict
import pandas as pd

data_root = Path.cwd().parent / "data"
participants_file = data_root / "TDBRAIN_participants_V3.xlsx"
part_df = pd.read_excel(participants_file)

neo_cols = [f'neoFFI_q{i}' for i in range(1, 61)]
filt_df = part_df[
    (part_df["formal_status"] != "UNKNOWN") &
    (part_df["age"].notna()) &
    (part_df["gender"].notna()) &
    (part_df["education"].notna()) &
    (part_df["n_oddb_CP"].notna()) &
    (part_df["n_oddb_FP"].notna()) &
    (part_df["n_oddb_CN"].notna()) &
    (part_df["n_oddb_FN"].notna()) &
    (part_df["avg_rt_oddb_CP"].notna()) &
    (part_df[neo_cols].notna().all(axis=1))
]

filt_ids = set(filt_df["TDBRAIN_ID"])
bdf_index = defaultdict(list)
for path in Path(data_root).rglob("*.bdf"):
    subj_id = path.name.split("_")[0]
    if subj_id in filt_ids and "ses-1" in path.parts:
        bdf_index[subj_id].append(path)

In [18]:
import mne
import json
mne.set_log_level('WARNING')

In [17]:
def process_bdf(bdf_path):
    """Processes a single bdf path from end to end, outputting the structured text that the LLM will see."""
    # stores the raw data in raw_bdf
    raw_bdf = mne.io.read_raw_bdf(bdf_path, preload=True)
    # creates a copy (to preserve the original raw signal) and only selects the EEG channels
    bdf = raw_bdf.copy().pick('eeg')
    # applies a bandpass filter (1-40 Hz) in order to remove movement artifacts and other noise
    bdf.filter(l_freq=1.0, h_freq=40.0)
    # transforms into signal power distributed across specific frequencies 
    spectrum = bdf.compute_psd(method='welch', fmin=1, fmax=40.0)
    # extract the psd data and the frequency ranges
    psds, freqs = spectrum.get_data(return_freqs=True)
    band_powers = {}
    # the frequency ranges and their corresponding band names
    freq_ranges = {"delta": (0.5, 4.0), "theta": (4.0, 8.0), "alpha": (8.0, 13.0), "beta": (13.0, 30.0), "gamma": (30.0, 100.0)}
    for band, (fmin, fmax) in freq_ranges.items():
        # create a boolean mask for the freq range in question
        mask = (freqs >= fmin) & (freqs <= fmax)
        # in the psds data, keeps all dimensions same (through the ...), apply mask to the last dim (keeping only the freqs within the mask)
        # and then taskes the mean across that very last dimension (using -1)
        # finally, the multiplication by 1e12 converts these into microvolts squared to increase LLM comprehension with the numbers
        band_powers[band] = psds[..., mask].mean(axis=-1) * 1e12
    return band_powers
process_bdf(bdf_index['sub-88045085'][0])

{'delta': array([ 3.41209032,  3.37670723,  3.87247367,  4.79215631,  5.17254274,
         4.70583359,  2.13645556,  4.53799485,  6.06413428,  4.49785626,
         2.83479036,  4.41168089,  4.79587086,  3.43651572,  2.81048342,
         4.01537949,  4.29836983,  3.4946388 ,  1.28098774,  3.39923461,
         4.93624339,  3.03185512,  1.72260738,  1.61132291,  1.66838253,
         1.65167423,  9.31835014, 10.03217307,  4.34483014,  4.32849261,
        22.15200633,  8.41857565]),
 'theta': array([ 1.42271407,  1.47361917,  1.6102064 ,  1.89674678,  2.23630531,
         1.91781457,  1.03955688,  1.77192529,  2.35210896,  1.75206885,
         1.08049141,  1.58656845,  2.00934858,  1.46463522,  0.76661218,
         1.46889638,  1.73842943,  1.45523805,  0.62062669,  1.30359765,
         1.65152357,  1.31906353,  0.80728604,  0.89720838,  0.96713237,
         0.95666355,  3.30465688,  1.98053395,  1.5885133 ,  1.46899062,
        31.15626643,  3.09510192]),
 'alpha': array([ 3.45630881,  3.6

In [43]:
print(len(bdf_index))
print(len(set(bdf_index.keys())))
print(filt_df["TDBRAIN_ID"].nunique())

355
355
360


In [44]:
print(len(filt_df))
print(filt_df["TDBRAIN_ID"].nunique())

375
360
